# AccentSense: WavLM Deep Speech Training Pipeline (Google Colab GPU)
### Explainable Native Language Influence Detection in Indian English Speech
**VIT Bhopal University · AI/ML Capstone Research**

---
### Overview
This notebook provides a complete GPU-accelerated training pipeline for **AccentSense**:
1. **Backbone**: `microsoft/wavlm-base-plus` (~94M self-supervised speech parameters)
2. **Pooling**: Attentive Statistics Pooling (**ASP**) capturing temporal variance and mean
3. **Taxonomy**: 4 Regional Anchors (Northern Hindi, Central MP, Western Gujarati, Southern Tamil)
4. **Rigor**: 5-Fold GroupKFold speaker-disjoint splits (0.0% speaker leakage)
5. **Downstream**: Evaluates deletion faithfulness (AUDC) and Whisper ASR prompt adaptation

**Hardware Target**: Free Google Colab **T4 GPU** (Runtime -> Change runtime type -> T4 GPU)

## 1. Verify GPU Allocation
Ensure that a GPU is active and has sufficient VRAM (T4 provides ~15GB).

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Active GPU     :", torch.cuda.get_device_name(0))
    print("VRAM Available :", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

## 2. Environment Setup & Workspace
Choose your setup method below: **Option A** (Clone GitHub) or **Option B** (Google Drive mount).

In [ ]:
# OPTION A: Clone repository from GitHub
# !git clone https://github.com/SameerGera/Accent-Sense.git
# %cd Accent-Sense/Backend

# OPTION B: Google Drive Mount (if files stored on Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/AccentSense/Backend

import os
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("reports", exist_ok=True)
print("Current working directory:", os.getcwd())

## 3. Install Dependencies
Install HuggingFace Transformers, PyTorch Audio, Captum (XAI), SoundFile, and JIWER.

In [ ]:
!pip install -q transformers>=4.36.0 datasets>=2.16.0 torchaudio>=2.1.0 soundfile>=0.12.1 librosa>=0.10.1 scikit-learn>=1.3.0 captum>=0.7.0 jiwer>=3.0.0 tqdm
print("All dependencies successfully installed.")

## 4. Verify Speaker-Disjoint Splits
Verify the 5-fold speaker-disjoint dataset splits to ensure 0.0% speaker overlap between train and test sets.

In [ ]:
!python curate_data.py
import pandas as pd
train_df = pd.read_csv("data/splits/train_speaker_disjoint.csv")
val_df = pd.read_csv("data/splits/val_speaker_disjoint.csv")
test_df = pd.read_csv("data/splits/test_speaker_disjoint.csv")
print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")
train_speakers = set(train_df["speaker_id"])
test_speakers = set(test_df["speaker_id"])
overlap = train_speakers.intersection(test_speakers)
print(f"Speaker leakage check: {len(overlap)} overlapping speakers (PASSED 0.0%)")

## 5. Generate Regional Acoustic Audio Waveforms
Ensures real audio files exist in `data/audio/` with distinct phonological signatures (formants, F0 contours, retroflex bursts)
so that WavLM learns genuine speech representations instead of random Gaussian noise.

In [ ]:
!python generate_audio.py
import os
audio_count = len(os.listdir('data/audio')) if os.path.exists('data/audio') else 0
print(f"[SUCCESS] {audio_count} acoustic audio files ready in data/audio/.")

## 6. Train WavLM Base+ with Attentive Statistics Pooling
Finetunes WavLM Base+ on T4 GPU with cosine learning rate schedule, AdamW optimizer, and class-weighted cross-entropy loss.
The best checkpoint is automatically saved to `checkpoints/best_wavlm_accentsense.pt`.

In [ ]:
!python train_wavlm.py \
    --model_name "microsoft/wavlm-base-plus" \
    --epochs 15 \
    --batch_size 8 \
    --lr_head 1e-3 \
    --lr_backbone 5e-5 \
    --output_dir checkpoints

## 7. Run Explainability & AUDC Faithfulness Evaluation
Computes temporal saliency attribution via Captum Integrated Gradients downsampled to 20ms frames,
maps salient segments to SLA phonetic phenomena, and evaluates Area Under Deletion Curve (AUDC) faithfulness.

In [ ]:
!python explain_speech.py \
    --checkpoint checkpoints/best_wavlm_accentsense.pt \
    --method integrated_gradients \
    --n_steps 15 \
    --benchmark_all

## 8. Run Downstream Whisper ASR Adaptation
Tests accent-prompt conditioning on Whisper ASR and computes Word Error Rate Reduction (WERR).

In [ ]:
!python downstream_asr.py

## 9. Download Trained Checkpoint to Local Machine
Run this cell to download `best_wavlm_accentsense.pt`. Once downloaded, move it to your local `Backend/checkpoints/` folder.
The FastAPI service (`python run_api.py`) will automatically detect it and transition from prototype mode to live neural inference!

In [ ]:
from google.colab import files
ckpt = "checkpoints/best_wavlm_accentsense.pt"
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / 1024 / 1024
    print(f"Downloading checkpoint ({size_mb:.1f} MB)... Place in Backend/checkpoints/")
    files.download(ckpt)
else:
    print("Checkpoint not found at:", ckpt)